## Install Dependencies

In [ ]:
!pip install langchain langchain-community langchain-huggingface langchain-chroma sentence-transformers rank_bm25

## Imports

In [ ]:
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

## Setup Retrievers (Keyword + Semantic)

In [ ]:
texts = ["Apples are red.", "Apples are green.", "Bananas are yellow."]

# 1. BM25 Keyword Retriever (Exact word matches)
bm25_retriever = BM25Retriever.from_texts(texts)
bm25_retriever.k = 2

# 2. Vector Semantic Retriever (Meaning matches)
vectorstore = Chroma.from_texts(texts, HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Combine using Reciprocal Rank Fusion (RRF)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever], weights=[0.5, 0.5]
)

## Hybrid Search Function

In [ ]:
def hybrid_search(query):
    return ensemble_retriever.invoke(query)

## Test Hybrid Retrieval

In [ ]:
docs = hybrid_search("yellow fruit")